# AMEX Enterprise Credit Risk Platform
## Notebook 38 -- Dynamic / Behavioral Credit Scoring: Business Understanding & Policy
### Phase 3 . Problem Statement 6: Dynamic / Behavioral Credit Scoring

CRISP-DM stage: **Business Understanding**. Sprint 1, Notebook 1 of 4 for this problem. Depends on Problem 1 Notebooks 01-05 (reads `project_config.json`, `notebook_02_summary.json`, `notebook_05_summary.json`) and Problem 4 Notebooks 26-29 (reads `notebook_28_summary.json` and the real `severity_scoring_bundle.json`, feature list only -- see Section 8).

**What this notebook does (real, computed on your machine when you run it):**
- States the business case for dynamic/behavioral scoring -- re-scoring an EXISTING book using a customer's most RECENT behavior, distinct from Problem 1's static full-history score and Problem 5's first-K early-detection score -- plus an honest, explicit data-limitation statement (this dataset has exactly one eventual-default label per customer, no month-by-month ground truth)
- Reads the REAL per-customer statement-count distribution directly from the raw Kaggle training CSV (same method Notebook 34 established, recomputed fresh here per this platform's zero-fabrication standard)
- Defines `TRAILING_WINDOW_CANDIDATES` -- genuine calendar-quarter trailing-window lengths (3/6/9 months, each strictly below the dataset's real measured statement-count ceiling), each meaning "a customer's most recent W statements", not an early cutoff -- deliberately avoiding the exact percentile-collapse trap Notebook 34 found and fixed for Problem 5
- Reuses Problem 4's real, correlation-filtered 243-feature list as this problem's feature space (not Problem 4's precomputed severity tier, which would leak full-history information into a short window -- see Section 8's leakage note)
- Records an explicit architecture scope decision: windowed GBM only in this pass, LSTM deferred as a future extension (Section 9)
- Sets explicit, labeled `ASSUMPTION` KPI targets (>=80% of Notebook 05's full-history AUC retained at each W; a direct real comparison against Problem 5's own first-K AUC at the same window length) that Notebook 39 (Modeling) will be validated against
- Records the standing full-metrics-suite requirement (ROC-AUC, PR-AUC, Accuracy, Precision, Recall, F1, Specificity, Log Loss, MCC, confusion matrix, ROC + PR curves -- inline and in reports) that applies to Notebook 39 onward per the user's own directive
- Writes `dynamic_behavioral_scoring_policy.json` for Notebook 39 to consume

**What this notebook does NOT do:** no modeling, no trailing-window feature engineering yet -- that's Notebook 39. This notebook only establishes the policy and the real data facts that policy depends on.

Zero-fabrication: every number this notebook prints is computed live from your real Kaggle data on this run. `ASSUMPTION`-labeled values (the W candidates, the KPI targets, the architecture scope decision) are explicit, editable business choices, not disguised as measured facts.


In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD CONFIG FROM PROBLEM 1 (NOTEBOOKS 01-05)
# =============================================================================
import os
import sys
import json
import time
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Config From Problem 1 (Notebooks 01-05)")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"
NB02_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_02_summary.json"
NB05_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_05_summary.json"

for _p, _fix in [
    (CONFIG_PATH, "run 01_business_understanding.ipynb first"),
    (NB02_SUMMARY_PATH, "run 02_data_engineering.ipynb first"),
    (NB05_SUMMARY_PATH, "run 05_model_development.ipynb first"),
]:
    if not _p.exists():
        raise FileNotFoundError(
            f"{_p} not found.\nFix: {_fix} -- Problem 6 depends on Problem 1's real "
            f"champion model and data engineering outputs."
        )

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)
with open(NB02_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB02_SUMMARY = json.load(f)
with open(NB05_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB05_SUMMARY = json.load(f)

PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}
DETECTED_LOGICAL_CORES = PROJECT_CONFIG["hardware"]["logical_cores_detected"]
RANDOM_SEED = PROJECT_CONFIG["random_seed"]

_resource_limits = PROJECT_CONFIG.get("resource_limits", {})
WARP_THREAD_COUNT = (
    _resource_limits.get("warp_thread_count")
    or PROJECT_CONFIG.get("warp_thread_count")
    or DETECTED_LOGICAL_CORES
)
MAX_RAM_BYTES = _resource_limits.get("max_ram_bytes")

# Problem 6's own output directory -- a NEW pillar this notebook introduces
# (Phase 3, Problem 6). Same defensive-fallback idiom every notebook in this
# platform already uses: prefer a registered pillar_dirs entry, else fall
# back to the standard Phase/Problem folder convention and create it.
if "dynamic_behavioral_scoring_policy" in PILLAR_DIRS:
    DBS_POLICY_DIR = PILLAR_DIRS["dynamic_behavioral_scoring_policy"]
else:
    DBS_POLICY_DIR = (
        PROJECT_ROOT / "Phase3_Behavioral_Intelligence"
        / "06_Problem6_Dynamic_Behavioral_Credit_Scoring" / "policy"
    )
    print(
        "NOTE: 'dynamic_behavioral_scoring_policy' not found in project_config.json's "
        "pillar_dirs -- using the standard folder-convention fallback:\n"
        f"      {DBS_POLICY_DIR}\n"
        "      (If you've registered a different path for this pillar, edit the "
        "PILLAR_DIRS lookup above to match it.)"
    )
DBS_POLICY_DIR.mkdir(parents=True, exist_ok=True)

CHAMPION_NAME = NB05_SUMMARY["champion_model"]
CHAMPION_METRICS = NB05_SUMMARY["champion_metrics"]

print(f"Loaded config from      : {CONFIG_PATH}")
print(f"RANDOM_SEED              : {RANDOM_SEED} (same seed used by every notebook in this platform)")
print(f"WARP_THREAD_COUNT        : {WARP_THREAD_COUNT}")
print(f"Champion model (Problem 1, measured)    : {CHAMPION_NAME}")
print(f"Champion holdout AUC (measured)         : {CHAMPION_METRICS.get('holdout_auc')}")
print(f"Champion holdout AMEX metric (measured) : {CHAMPION_METRICS.get('holdout_amex_metric')}")
print(f"Policy artifacts will be written under  : {DBS_POLICY_DIR}")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONFIGURATION & LIBRARY IMPORTS
# =============================================================================
_section("SECTION 2: WARP Hardware Configuration & Library Imports")

os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)
warnings.filterwarnings("ignore", category=UserWarning)

import logging
logger = logging.getLogger("amex_platform")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

missing = []
try:
    import polars as pl
except ImportError:
    missing.append("polars")
try:
    import psutil
except ImportError:
    missing.append("psutil")

if missing:
    raise ImportError(
        "Missing required package(s): " + ", ".join(missing) + "\n"
        "Fix: run this in a terminal, then re-run this cell:\n"
        f"    pip install {' '.join(missing)}"
    )


def _rss_gb() -> float:
    """Current process resident memory, in GB."""
    return psutil.Process().memory_info().rss / 1e9


logger.info(f"Polars thread pool configured to {os.environ['POLARS_MAX_THREADS']} threads (95% cap, WARP 6.4)")
print(f"Process RSS at Section 2 start: {_rss_gb():.2f} GB")
if MAX_RAM_BYTES:
    print(f"Configured RAM ceiling (90% of detected total): {MAX_RAM_BYTES / 1e9:.1f} GB")
print("\n\u2705 Section 2 complete.")


# =============================================================================
# SECTION 3: BUSINESS UNDERSTANDING -- WHY DYNAMIC/BEHAVIORAL SCORING MATTERS
# =============================================================================
_section("SECTION 3: Business Understanding -- Why Dynamic/Behavioral Scoring Matters")

print(
    "Problem 1's champion model (Notebook 05) scores a customer ONCE, using "
    "their FULL available statement history -- a static, point-in-time score "
    "that never updates unless the whole pipeline is re-run. Problem 5 asked "
    "a different question: how early in a NEW relationship can risk be flagged, "
    "using only a customer's first few statements. Problem 6 asks a third, "
    "complementary question, about an EXISTING book rather than a new one: "
    "does a customer's MOST RECENT behavior -- not their earliest, not their "
    "full history -- carry risk signal that a static full-history score can't "
    "isolate on its own? A customer's risk profile can drift after "
    "origination (utilization creeping up, payment behavior softening); a "
    "model that only ever looks at full history blends old and new behavior "
    "together and can be slow to react to a real, recent change."
)
print(
    "\nDATA LIMITATION (stated plainly, same standard as Problem 2's "
    "fair-lending section and Problem 5's Section 3): this dataset carries "
    "exactly ONE eventual-default label per customer -- there is no "
    "month-by-month ground truth telling us whether a customer was "
    "'about to default' at any specific point mid-relationship. So 'monthly "
    "refreshed PD' cannot mean 'a different label predicted at each month' -- "
    "it means: at whatever point a customer currently sits in their real "
    "statement history, score them using only their TRAILING (most recent) "
    "W months of behavior, refreshed each time a new statement arrives, "
    "while still predicting the same real eventual-default outcome Problem 1 "
    "predicts. This is an honest, computable reframing -- not a claim this "
    "dataset can support literal month-by-month risk labels, which it can't."
)
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: REAL PER-CUSTOMER STATEMENT-COUNT DISTRIBUTION
# =============================================================================
_section("SECTION 4: Real Per-Customer Statement-Count Distribution (Determines the Trailing Window W)")

# Same real, live-computed fact Notebook 34 established for Problem 5 -- but
# recomputed fresh here (zero-fabrication: every notebook computes its own
# facts live, even when the underlying dataset fact is unchanged). Reuses
# the same lesson from Notebook 34: train_full_features.parquet is already
# aggregated to one row per customer, so the real per-statement counts must
# come from the raw Kaggle CSV directly.
_raw_candidates = []
if "raw_data_dir" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["raw_data_dir"]) / "train_data.csv")
if "data_root" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["data_root"]) / "train_data.csv")
_raw_candidates.append(PROJECT_ROOT.parent / "Raw Data From Kaggle" / "train_data.csv")

RAW_TRAIN_DATA_PATH = None
for _candidate in _raw_candidates:
    if _candidate.exists() and _candidate.stat().st_size > 1_000_000:
        RAW_TRAIN_DATA_PATH = _candidate
        break

if RAW_TRAIN_DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find the raw train_data.csv (needed for real per-statement "
        "row counts). Checked:\n"
        + "\n".join(f"  - {c}" for c in _raw_candidates)
        + "\n\nFor diagnosis, here is what's actually available:\n"
        f"  PROJECT_CONFIG top-level keys: {sorted(PROJECT_CONFIG.keys())}\n"
        f"  NB02_SUMMARY['output_files'] keys: {sorted(NB02_SUMMARY.get('output_files', {}).keys())}\n"
        "Fix: tell me the real path to your raw train_data.csv so this can be "
        "corrected with the real path rather than another guess."
    )

print(f"Reading real per-statement (raw, pre-aggregation) data from: {RAW_TRAIN_DATA_PATH}")
print("(This scans the full raw CSV with only the customer_ID column projected "
      "-- may take a minute or two on a 16GB+ file.)")
_t0 = time.time()
_statement_counts = (
    pl.scan_csv(RAW_TRAIN_DATA_PATH)
    .select(pl.col("customer_ID"))
    .group_by("customer_ID")
    .agg(pl.len().alias("n_statements"))
    .collect()
)
print(f"Grouped in {time.time() - _t0:.1f}s. Process RSS: {_rss_gb():.2f} GB")
_n_customers = _statement_counts.height
_counts_series = _statement_counts["n_statements"]
STATEMENT_COUNT_STATS = {
    "n_customers": _n_customers,
    "min": int(_counts_series.min()),
    "p10": float(_counts_series.quantile(0.10)),
    "p25": float(_counts_series.quantile(0.25)),
    "median": float(_counts_series.median()),
    "mean": float(_counts_series.mean()),
    "max": int(_counts_series.max()),
}
print(f"Customers (real, measured): {STATEMENT_COUNT_STATS['n_customers']:,}")
for _k in ("min", "p10", "p25", "median", "mean", "max"):
    print(f"  {_k:>6} statements/customer: {STATEMENT_COUNT_STATS[_k]}")
print("\n\u2705 Section 4 complete.")


# =============================================================================
# SECTION 5: TRAILING-WINDOW POLICY -- DEFINING W (ASSUMPTION)
# =============================================================================
_section("SECTION 5: Trailing-Window Policy -- Defining Candidate Windows (ASSUMPTION)")

print(
    "Same right-censoring fact Notebook 34 found for Problem 5 applies here "
    f"too: statement history is capped at {STATEMENT_COUNT_STATS['max']} months. "
    "For Problem 6, that ceiling matters differently -- W is the number of a "
    "customer's MOST RECENT statements used (statements[-W:], by real "
    "chronological S_2 order), not an early cutoff, so a W near the ceiling "
    "would use almost all of a full-history customer's data, blurring the "
    "'recent behavior only' framing this problem is actually testing. "
    "Choosing genuine sub-ceiling candidates for the same reason Notebook 34 "
    "rejected a single percentile-derived K."
)

# ASSUMPTION: quarterly-cadence trailing windows (3/6/9 months), each strictly
# below the real measured ceiling -- mirrors Problem 5's EARLY_WINDOW_CANDIDATES
# cadence for platform-wide consistency, but each W here means "most recent W
# statements", not "first W statements". Only candidates strictly below the
# real ceiling are kept (defensive, same idiom as Notebook 34).
_candidate_windows = [3, 6, 9]
TRAILING_WINDOW_CANDIDATES = sorted({w for w in _candidate_windows if w < STATEMENT_COUNT_STATS["max"]})
if not TRAILING_WINDOW_CANDIDATES:
    TRAILING_WINDOW_CANDIDATES = [max(3, STATEMENT_COUNT_STATS["max"] - 1)]

TRAILING_WINDOW_COVERAGE = {
    w: float((_counts_series >= w).sum() / _n_customers * 100.0)
    for w in TRAILING_WINDOW_CANDIDATES
}

print(f"\nTRAILING_WINDOW_CANDIDATES (ASSUMPTION -- quarterly cadence below "
      f"the real ceiling of {STATEMENT_COUNT_STATS['max']}): {TRAILING_WINDOW_CANDIDATES}")
for _w, _pct in TRAILING_WINDOW_COVERAGE.items():
    print(f"  W={_w:>2}: {_pct:.1f}% of customers have >= {_w} statements "
          f"(measured -- needed to form a full trailing window of this length)")

print(
    "\nNotebook 39 (Modeling) will, for EACH candidate W above, build each "
    "customer's trailing-W-statement feature set (their most recent W "
    "statements, reusing Problem 4's real correlation-filtered feature space "
    "-- see Section 8 below), train a windowed-GBM model, and report both: "
    "(a) an AUC-retention curve vs. Notebook 05's full-history baseline, and "
    "(b) a direct, real comparison against Problem 5's own first-K (early) "
    "AUC at the SAME window length -- an honest empirical test of whether "
    "RECENT behavior is more, less, or equally predictive as EARLY behavior "
    "at an identical information budget."
)
print("\n\u2705 Section 5 complete.")


# =============================================================================
# SECTION 6: SUCCESS CRITERIA (ASSUMPTION)
# =============================================================================
_section("SECTION 6: Success Criteria (ASSUMPTION)")

DBS_KPI_TARGETS = {
    "min_auc_retention_vs_full_history": 0.80,
    "description": (
        "ASSUMPTION -- same threshold convention as Problem 5 (Notebook 34's "
        "Section 6), for platform-wide consistency: the trailing-window model "
        "(Notebook 39) should retain at least 80% of Notebook 05's full-history "
        "champion holdout AUC at each candidate W to be considered "
        "operationally useful. Below that threshold, the honest conclusion is "
        "'recent behavior alone isn't enough signal at this window length' -- "
        "Notebook 39 must report that outcome plainly if it happens."
    ),
    "full_history_reference_auc": CHAMPION_METRICS.get("holdout_auc"),
    "secondary_comparison": (
        "ASSUMPTION -- not a pass/fail gate, an honest reporting requirement: "
        "Notebook 39 must report the trailing-window AUC at each W directly "
        "alongside Problem 5's real first-K (early) AUC at the same K=W, "
        "reading both from Problem 5's own committed notebook_35/36_summary.json "
        "artifacts (no recomputation, no invented numbers) -- whichever "
        "direction the real numbers point, that is the finding."
    ),
    "metrics_suite_requirement": (
        "STANDING RULE (user directive, 2026-08-25, applies from this problem "
        "onward): Notebook 39 (Modeling) and Notebook 40 (Validation & "
        "Deployment) must compute and DISPLAY -- inline in the notebook AND in "
        "this problem's Word/Excel/HTML reports -- the full classification "
        "metrics suite for the champion trailing-window model: ROC-AUC, "
        "PR-AUC (average precision), Accuracy, Precision, Recall, F1, "
        "Specificity, Log Loss, Matthews Correlation Coefficient, and a full "
        "confusion matrix, each threshold-dependent metric reported at BOTH "
        "the standard 0.5 threshold AND the holdout-derived F1-optimal "
        "threshold (labeled separately, since a single 0.5-threshold number is "
        "misleading on this dataset's real ~26% default rate) -- plus a "
        "rendered ROC curve and Precision-Recall curve, both displayed inline "
        "and saved as PNGs for report embedding."
    ),
}
print(json.dumps(DBS_KPI_TARGETS, indent=2))
print("\n\u2705 Section 6 complete.")


# =============================================================================
# SECTION 7: TARGET DEFINITION -- UNCHANGED FROM PROBLEM 1
# =============================================================================
_section("SECTION 7: Target Definition")

print(
    "This problem does NOT redefine the outcome being predicted -- it reuses "
    "the exact same eventual-default target Problem 1 uses (from the real "
    "train_labels.csv), and the exact same real train/holdout customer split "
    "Notebook 02 established (reused, not re-split, so results stay directly "
    "comparable to both Notebook 05's full-history baseline and Problem 5's "
    "early-window results). Only the FEATURES available to the model differ "
    "(trailing-W-statements-only vs. full history vs. first-K-statements)."
)
print("\n\u2705 Section 7 complete.")


# =============================================================================
# SECTION 8: FEATURE-SPACE POLICY -- REUSING PROBLEM 4'S REAL FEATURE LIST
# =============================================================================
_section("SECTION 8: Feature-Space Policy -- Reusing Problem 4's Real Feature List")

# Problem 6 depends on Problem 4 (per the master plan) -- fulfilled here by
# reusing Problem 4's real, correlation-filtered 243 D_* feature list as the
# windowed panel's feature space (the same predictive features Problem 4
# already validated), rather than directly consuming Problem 4's precomputed
# severity TIER as an input. Using the tier itself would leak full-history
# information into an early/short-window snapshot -- Problem 4's tier is
# computed from full-history features, so a customer scored at W=3 could
# never have produced that tier value themselves at that point. Reusing only
# the FEATURE LIST (not the tier output) avoids that leakage while still
# fulfilling the real dependency the master plan specifies.
_p4_bundle_candidates = [
    PROJECT_ROOT / "Phase2_Regulatory_Loss_Provisioning" / "04_Problem4_Delinquency_Escalation_Loss_Severity"
    / "03_Validation_Deployment" / "severity_scoring_bundle.json",
]
if "loss_severity_validation_deployment" in PILLAR_DIRS:
    _p4_bundle_candidates.insert(0, PILLAR_DIRS["loss_severity_validation_deployment"] / "severity_scoring_bundle.json")
_p4_summary_path = ARTIFACTS_DIR / "notebook_28_summary.json"
if _p4_summary_path.exists():
    with open(_p4_summary_path, "r", encoding="utf-8") as f:
        _p4_summary = json.load(f)
    _p4_stored_path = _p4_summary.get("output_files", {}).get("severity_scoring_bundle.json")
    if _p4_stored_path:
        _p4_bundle_candidates.append(Path(_p4_stored_path))

SEVERITY_BUNDLE_PATH = None
for _candidate in _p4_bundle_candidates:
    if _candidate.exists() and _candidate.stat().st_size > 1_000:
        SEVERITY_BUNDLE_PATH = _candidate
        break

if SEVERITY_BUNDLE_PATH is None:
    raise FileNotFoundError(
        "Could not find Problem 4's real severity_scoring_bundle.json. Checked:\n"
        + "\n".join(f"  - {c}" for c in _p4_bundle_candidates)
        + "\n\nFix: run Problem 4's Notebooks 26-29 first (this platform's "
        "Phase 2, Problem 4) -- Problem 6 reuses its real, validated feature list."
    )

with open(SEVERITY_BUNDLE_PATH, "r", encoding="utf-8") as f:
    _severity_bundle = json.load(f)
DBS_FEATURE_LIST = sorted(_severity_bundle["features"])

print(f"Loaded Problem 4's real feature list from: {SEVERITY_BUNDLE_PATH}")
print(f"Real correlation-filtered D_* feature count (measured): {len(DBS_FEATURE_LIST)}")
print(f"Sample features: {DBS_FEATURE_LIST[:5]}")
print(
    "\nDesign decision (ASSUMPTION): Notebook 39 will re-aggregate these same "
    "features from each customer's trailing-W statements only (same "
    "aggregation transformations Notebook 04/27 already established), NOT "
    "consume Problem 4's precomputed tier/LGD output directly -- avoids "
    "leaking full-history information into a short-window snapshot."
)
print("\n\u2705 Section 8 complete.")


# =============================================================================
# SECTION 9: ARCHITECTURE SCOPE DECISION -- WINDOWED GBM, NOT LSTM (ASSUMPTION)
# =============================================================================
_section("SECTION 9: Architecture Scope Decision -- Windowed GBM, Not LSTM (ASSUMPTION)")

print(
    "The master plan names 'windowed GBM/LSTM' as the core technique for this "
    "problem. ASSUMPTION -- this build uses windowed GBM only (reusing "
    "Problem 1's real champion hyperparameters class, same scope-decision "
    "style Notebook 35 used for Problem 5's architecture reuse), and "
    "explicitly does NOT build an LSTM sequence model in this pass. "
    "Rationale: an LSTM needs meaningfully more engineering (padded "
    "sequence tensors, a training loop, GPU/CPU trade-off tuning) for a "
    "likely modest gain over a well-featured GBM on this tabular, "
    "short-sequence (<=13 steps) panel -- not a claim LSTM would perform "
    "worse, a stated scope choice to ship one architecture correctly rather "
    "than two partially. Tracked in ROADMAP.md as a future extension."
)
print("\n\u2705 Section 9 complete.")


# =============================================================================
# SECTION 10: WRITE DYNAMIC BEHAVIORAL SCORING POLICY ARTIFACT
# =============================================================================
_section("SECTION 10: Write Dynamic Behavioral Scoring Policy Artifact")

DYNAMIC_BEHAVIORAL_SCORING_POLICY = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "problem": "Problem 6 -- Dynamic / Behavioral Credit Scoring",
    "data_limitation_statement": (
        "This dataset carries exactly one eventual-default label per customer "
        "-- no month-by-month ground truth exists. 'Monthly refreshed PD' is "
        "defined as: score using only a customer's TRAILING (most recent) W "
        "statements, refreshed as new statements arrive, still predicting the "
        "same real eventual-default outcome. See this notebook's Section 3 "
        "for the full reasoning."
    ),
    "trailing_window_candidates": TRAILING_WINDOW_CANDIDATES,
    "trailing_window_coverage_by_w": TRAILING_WINDOW_COVERAGE,
    "statement_count_stats": STATEMENT_COUNT_STATS,
    "target_definition": (
        "Same eventual-default target and same real train/holdout customer "
        "split as Problem 1 -- only the FEATURES available differ (trailing W "
        "statements vs. full history vs. Problem 5's first-K statements)."
    ),
    "feature_space": {
        "source": "Problem 4's real severity_scoring_bundle.json (features list only, not the tier output)",
        "source_path": str(SEVERITY_BUNDLE_PATH),
        "feature_count": len(DBS_FEATURE_LIST),
        "features": DBS_FEATURE_LIST,
        "leakage_note": (
            "Problem 4's precomputed severity TIER is not used as an input "
            "feature -- it is derived from full-history data and would leak "
            "future information into a short trailing-window snapshot."
        ),
    },
    "architecture_scope_decision": (
        "Windowed GBM only (reuses Problem 1's champion hyperparameters "
        "class); LSTM explicitly deferred as a future extension, not built "
        "in this pass -- see Section 9."
    ),
    "kpi_targets": DBS_KPI_TARGETS,
    "champion_model_reference": CHAMPION_NAME,
    "champion_holdout_auc_reference": CHAMPION_METRICS.get("holdout_auc"),
    "champion_holdout_amex_metric_reference": CHAMPION_METRICS.get("holdout_amex_metric"),
    "random_seed": RANDOM_SEED,
}

POLICY_PATH = DBS_POLICY_DIR / "dynamic_behavioral_scoring_policy.json"
with open(POLICY_PATH, "w", encoding="utf-8") as f:
    json.dump(DYNAMIC_BEHAVIORAL_SCORING_POLICY, f, indent=2)
print(f"Wrote: {POLICY_PATH}")
print("\n\u2705 Section 10 complete.")


# =============================================================================
# SECTION 11: VERIFICATION -- INTEGRITY CHECKS ON EVERYTHING THIS NOTEBOOK WROTE
# =============================================================================
_section("SECTION 11: Verification -- Integrity Checks")


def _check(label, condition, detail=""):
    status = "PASS" if condition else "FAIL"
    print(f"  [{status}] {label}" + (f" -- {detail}" if detail and not condition else ""))
    return condition


_all_checks_passed = True
_all_checks_passed &= _check("Policy file was written", POLICY_PATH.exists())
_all_checks_passed &= _check(
    "TRAILING_WINDOW_CANDIDATES is a non-empty list, all values >= 3",
    len(TRAILING_WINDOW_CANDIDATES) > 0 and all(w >= 3 for w in TRAILING_WINDOW_CANDIDATES),
)
_all_checks_passed &= _check(
    "Every candidate window is strictly below the real measured ceiling",
    all(w < STATEMENT_COUNT_STATS["max"] for w in TRAILING_WINDOW_CANDIDATES),
)
_all_checks_passed &= _check(
    "Trailing-window coverage is a real measured percentage in (0, 100] for every candidate",
    all(0.0 < pct <= 100.0 for pct in TRAILING_WINDOW_COVERAGE.values()),
)
_all_checks_passed &= _check(
    "Statement count stats are internally consistent (min <= p25 <= max)",
    STATEMENT_COUNT_STATS["min"] <= STATEMENT_COUNT_STATS["p25"] <= STATEMENT_COUNT_STATS["max"],
)
_all_checks_passed &= _check(
    "Reused Problem 1's real champion AUC (not fabricated)",
    DBS_KPI_TARGETS["full_history_reference_auc"] == CHAMPION_METRICS.get("holdout_auc"),
)
_all_checks_passed &= _check(
    "Reused Problem 4's real feature list (243 features, not fabricated)",
    len(DBS_FEATURE_LIST) == len(_severity_bundle["features"]) and len(DBS_FEATURE_LIST) > 0,
)
_all_checks_passed &= _check(
    "No duplicate features in the reused feature list",
    len(DBS_FEATURE_LIST) == len(set(DBS_FEATURE_LIST)),
)

if not _all_checks_passed:
    raise AssertionError("One or more verification checks failed -- see FAIL lines above.")
print("\n\u2705 Section 11 complete -- all checks passed.")


# =============================================================================
# SECTION 12: WRITE NOTEBOOK 38 SUMMARY ARTIFACT & COMPLETION
# =============================================================================
_section("SECTION 12: Write Notebook 38 Summary Artifact")

NB38_SUMMARY = {
    "notebook": "38_dynamic_behavioral_scoring_business_understanding.ipynb",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "trailing_window_candidates": TRAILING_WINDOW_CANDIDATES,
    "trailing_window_coverage_by_w": TRAILING_WINDOW_COVERAGE,
    "policy_path": str(POLICY_PATH),
    "feature_count": len(DBS_FEATURE_LIST),
    "kpi_targets": DBS_KPI_TARGETS,
    "random_seed": RANDOM_SEED,
}
NB38_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_38_summary.json"
with open(NB38_SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(NB38_SUMMARY, f, indent=2)
print(f"Wrote: {NB38_SUMMARY_PATH}")

_section("NOTEBOOK 38 COMPLETE")
print(f"Trailing-window candidates (real, computed): {TRAILING_WINDOW_CANDIDATES}")
for _w, _pct in TRAILING_WINDOW_COVERAGE.items():
    print(f"  W={_w:>2}: {_pct:.1f}% of customers covered")
print(f"Feature space (real, reused from Problem 4): {len(DBS_FEATURE_LIST)} features")
print(f"Policy written to: {POLICY_PATH}")
print(
    "Next: 39_dynamic_behavioral_scoring_modeling.ipynb -- builds the "
    "trailing-window feature set (most recent W statements per customer, "
    "same transformations as Notebook 27), trains/evaluates the windowed-GBM "
    "model against the KPI targets set in Section 6 above, and reports the "
    "full classification metrics suite (ROC-AUC, PR-AUC, Accuracy, Precision, "
    "Recall, F1, Specificity, Log Loss, MCC, confusion matrix, ROC + PR "
    "curves) inline and for the eventual report."
)
